# Ownership level × disagreement

Two families that the count-based signals cannot express, split into their own fast module because neither needs a lag, a strict quarter join, or a time-series z-score — they are pure cross-sectional measures at *t*.

**Ownership level** — *how much* of the company these institutions hold:

| | |
|---|---|
| `inst_own` | `Σ position_value / market_cap` — the collective stake |
| `max_own` | the largest single holder's stake |
| `mean_own` | average stake |
| `top5_own` | five largest stakes combined |
| `hhi_own` | `Σ stake²` — Herfindahl concentration |

**Disagreement** — how *differently* funds position in the name:

| | |
|---|---|
| `disp_aw` | sd of `active_weight` across holders |
| `sum_abs_aw` | `Σ \|active_weight\|` — total tilt in either direction |
| `mean_abs_aw` | the same, per holder |
| `disp_z_aw` | sd of the within-fund z — removes each fund's own scale |
| `disp_aw_cv` | `disp_aw / mean\|active_weight\|` — relative disagreement |

Why these differ from `n_funds`: **3 funds holding 10% each** outrank **300 funds holding 0.01% apiece**. A headcount says the opposite. And unlike the counts, all of these are continuous — no ties, so the `tie_break` problem is irrelevant.

The question this notebook is built around: **does the ownership signal work better where funds disagree?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import own_disp as OD

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

# ROOT contains manager_holdings/ and return_data_v2.csv
ROOT = '.'
cfg = OD.Config(root=ROOT, tie_break='random', size_groups=3)

sq = OD.build(cfg)
sq.head(3)

In [ ]:
# Are these really distinct from each other, and from the holder count?
cols = ['n_funds'] + OD.OWN_SIGNALS + OD.DISP_SIGNALS
cols = [c for c in cols if c in sq.columns]
C = sq[cols].corr()

print(f'corr(n_funds, inst_own) = {C.loc["n_funds", "inst_own"]:+.3f}')
print(f'corr(n_funds, max_own)  = {C.loc["n_funds", "max_own"]:+.3f}')
print(f'corr(inst_own, disp_aw) = {C.loc["inst_own", "disp_aw"]:+.3f}'
      '   <- if high, the 5x5 corners will be nearly empty\n')

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=8)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f'{C.iloc[i, j]:.2f}', ha='center', va='center', fontsize=6,
                color='white' if abs(C.iloc[i, j]) > 0.6 else 'black')
ax.set_title('count  vs  ownership level  vs  disagreement')
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 1. Each signal on its own

In [ ]:
perf = OD.evaluate(sq, cfg)
perf.to_csv('own_disp_performance.csv', index=False)

# discovery vs held-out validation -- the only test that matters
p = (perf[perf['sample'] != 'all']
     .pivot_table(index=['signal', 'horizon'], columns='sample', values='t_nw'))
print('\nt by sample (sign flip = dead):')
p.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))

d = perf[perf['sample'] == 'all']
for sig, g in d.groupby('signal'):
    g = g.sort_values('horizon')
    ax[0].plot(g.horizon, g.t_nw, marker='o', lw=1.5,
               ls='--' if sig in OD.DISP_SIGNALS else '-', label=sig)
for y in (-2, 2):
    ax[0].axhline(y, color='r', ls=':', lw=0.9)
ax[0].axhline(0, color='k', lw=0.7)
ax[0].set_xticks([1, 2, 3]); ax[0].set_xlabel('horizon (quarters ahead)')
ax[0].set_ylabel('Newey-West t')
ax[0].set_title('solid = ownership, dashed = disagreement')
ax[0].legend(fontsize=7, ncol=2)

if {'discovery', 'validation'}.issubset(p.columns):
    ok = np.sign(p.discovery) == np.sign(p.validation)
    ax[1].scatter(p.discovery[ok], p.validation[ok], s=45, c='#2CA02C', label='same sign')
    ax[1].scatter(p.discovery[~ok], p.validation[~ok], s=45, c='#D62728', label='SIGN FLIP')
    lim = np.nanmax(np.abs(p.to_numpy())) * 1.2
    ax[1].plot([-lim, lim], [-lim, lim], 'k--', lw=0.8)
    ax[1].axhline(0, color='k', lw=0.6); ax[1].axvline(0, color='k', lw=0.6)
    ax[1].set_xlabel('t, discovery'); ax[1].set_ylabel('t, validation')
    ax[1].set_title('does it survive out of sample?'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. The 5 × 5 matrix

Independent sorts: each quarter the cross-section is bucketed on `own` and, separately, on `disp`. A stock lands in the cell given by both.

**Read the `n` matrix first.** If the two signals are correlated the corner cells go nearly empty, and a cell holding 3 stocks is not interpretable no matter what its *t* says.

**Then read the margins, not the cells.** Every cell's mean return is a *level*, so its *t* mostly reflects that equities went up — it is not alpha. The long-short margins are the informative part:

- `own_ls` — high-minus-low **ownership**, computed *within* each disagreement bucket. **This is the question**: if its *t* rises with the disagreement bucket, ownership predicts better where funds disagree.
- `disp_ls` — high-minus-low **disagreement** within each ownership bucket.

In [ ]:
OWN, DISP, H = 'inst_own', 'disp_aw', 1

ds = OD.double_sort(sq, OWN, DISP, n_own=5, n_disp=5, cfg=cfg, horizon=H)
OD.print_double_sort(ds, OWN, DISP)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

for a, key, title, fmt in [(ax[0], 'mean', 'mean forward return', '{:.3f}'),
                           (ax[1], 't', 'cell t-stat (a LEVEL, not alpha)', '{:.1f}'),
                           (ax[2], 'n', 'avg stocks per cell', '{:.0f}')]:
    M = ds[key]
    cmap = 'RdBu_r' if key != 'n' else 'viridis'
    v = np.nanmax(np.abs(M.to_numpy())) if key != 'n' else None
    im = a.imshow(M, cmap=cmap, **({'vmin': -v, 'vmax': v} if v else {}))
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            a.text(j, i, fmt.format(M.iloc[i, j]), ha='center', va='center',
                   fontsize=8)
    a.set_xlabel(f'{DISP} bucket  (0 = most agreement)')
    a.set_ylabel(f'{OWN} bucket  (0 = least owned)')
    a.set_title(title)
    plt.colorbar(im, ax=a, fraction=0.046)
plt.suptitle(f'{OWN} x {DISP},  horizon {H}')
plt.tight_layout(); plt.show()

thin = (ds['n'] < 10).sum().sum()
if thin:
    print(f'[warn] {thin} of {ds["n"].size} cells hold fewer than 10 stocks on '
          f'average -- those cells are not interpretable')

In [ ]:
# THE question: does ownership pay off more where funds disagree?
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for a, tbl, xcol, lab in [
        (ax[0], ds['own_ls'], 'disp_bucket', f'{OWN} high-low, within {DISP} bucket'),
        (ax[1], ds['disp_ls'], 'own_bucket', f'{DISP} high-low, within {OWN} bucket')]:
    if not len(tbl):
        continue
    cols = ['#4C78A8' if abs(t) >= 2 else '#C8C8C8' for t in tbl.t_nw]
    a.bar(tbl[xcol], tbl.t_nw, color=cols)
    for x, t in zip(tbl[xcol], tbl.t_nw):
        a.text(x, t, f'{t:.2f}', ha='center',
               va='bottom' if t >= 0 else 'top', fontsize=8)
    for y in (-2, 2):
        a.axhline(y, color='r', ls=':', lw=0.9)
    a.axhline(0, color='k', lw=0.8)
    a.set_xlabel(xcol); a.set_ylabel('Newey-West t'); a.set_title(lab, fontsize=10)

plt.suptitle('a monotone gradient is the evidence -- one significant bucket out of 5 is not')
plt.tight_layout(); plt.show()

if len(ds['own_ls']) >= 3:
    rho = ds['own_ls'].disp_bucket.corr(ds['own_ls'].t_nw, method='spearman')
    print(f'Spearman(disp bucket, t of {OWN} spread) = {rho:+.2f}')
    print('  positive => ownership predicts better where funds DISAGREE')

## 3. Every (own × disp) pair, and whether the gradient is monotone

`conditional_sort` differs from the matrix above: it runs the full **decile** long-short on the ownership signal *inside* each disagreement bucket, rather than comparing bucket means. `monotonicity` then asks whether the effect trends across buckets — a gradient is evidence, one lucky bucket out of five is not.

In [ ]:
grid = OD.conditional_grid(sq, cfg=cfg, n_buckets=5, horizons=(1, 2))
grid.to_csv('own_disp_conditional.csv', index=False)

mono = OD.monotonicity(grid)
print('sorted by |gap| = |t(highest bucket) - t(lowest bucket)|\n')
mono.round(3)

In [ ]:
# One panel per ownership signal: t across disagreement buckets.
conds = sorted(grid.condition.unique())
sigs = sorted(grid.signal.unique())
fig, axes = plt.subplots(1, len(sigs), figsize=(3.1 * len(sigs), 3.6),
                         sharey=True)
axes = np.atleast_1d(axes)
d1 = grid[(grid['sample'] == 'all') & (grid.horizon == 1)]
for a, sig in zip(axes, sigs):
    for cond in conds:
        g = d1[(d1.signal == sig) & (d1.condition == cond)].sort_values('bucket')
        if len(g):
            a.plot(g.bucket, g.t_nw, marker='o', lw=1.4, label=cond)
    for y in (-2, 2):
        a.axhline(y, color='r', ls=':', lw=0.8)
    a.axhline(0, color='k', lw=0.7)
    a.set_title(sig, fontsize=9); a.set_xlabel('condition bucket')
axes[0].set_ylabel('t of the long-short')
axes[-1].legend(fontsize=7)
plt.suptitle('ownership signal strength across disagreement buckets (h1)')
plt.tight_layout(); plt.show()

## How to read all of this

**The interaction is real** only if:

1. the `own_ls` *t* trends **monotonically** across disagreement buckets (Spearman well away from 0), not one bucket spiking;
2. every cell involved holds enough stocks — check the `n` matrix, ignore anything under ~10;
3. it survives at **h2**, which with a 45–60 day filing delay is the honest tradable window (h1 is only half-capturable);
4. the sign agrees between **discovery and validation**.

**Traps specific to this notebook:**

- Cell *t*-stats in the 5×5 matrix are large because equities drift up — they measure a level, not skill. Only the margins are long-short.
- Independent double sorts leave sparse corners whenever `own` and `disp` correlate. A dependent (sequential) sort would fix the counts but changes what is being asked.
- `disp_aw` grows mechanically with the size of the tilts; `disp_z_aw` and `disp_aw_cv` are the scale-free versions. If a result appears only in `disp_aw`, it is a magnitude effect, not disagreement.
- 5 buckets × 5 signals × 3 conditions × 2 horizons is 150 tests. At p<0.05 roughly 7 land by chance, which is why the gradient matters more than any single cell.

---
## 4. One focused strategy: ownership *within* the most-agreed names

Long the highest `inst_own` decile, short the lowest — but **only inside the lowest `sum_abs_aw` bucket**, i.e. the names funds collectively tilt on the least.

> **This combination was picked after looking at the results above.** It is one cell of a 5-bucket × several-signal grid, so its *t* is not a clean p-value. The discovery/validation split below is the check that matters: the split was fixed before any of this, and the validation half was never used to choose the cell.

In [ ]:
SIG, COND, BKT, HZ = 'inst_own', 'sum_abs_aw', 0, 1   # BKT=0 -> LOWEST bucket

sp = OD.strategy_spread(sq, SIG, COND, bucket=BKT, n_buckets=5,
                        cfg=cfg, horizon=HZ)
rep = OD.report_strategy(sp, cfg, HZ, f'long-short {SIG} | {COND} bucket {BKT}')
print(f'long TOP {SIG} decile, short BOTTOM, inside the LOWEST {COND} bucket, h{HZ}')
print(f'{len(sp)} quarters: {sp.index.min()} .. {sp.index.max()}\n')
rep.round(4)

In [ ]:
split = pd.Period(cfg.split, freq='Q')
s = sp.sort_index()
w = (1 + s).cumprod()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4),
                       gridspec_kw={'width_ratios': [2, 1]})

ax[0].plot(w.index.to_timestamp(), w.to_numpy(), lw=1.8, color='#4C78A8')
ax[0].axvline(split.to_timestamp(), color='k', ls=':', lw=1.2)
ax[0].axhline(1, color='k', lw=0.7)
ax[0].fill_between(w.index.to_timestamp(), 1, w.to_numpy(),
                   where=(w.to_numpy() >= 1), alpha=0.15, color='#2CA02C')
ax[0].fill_between(w.index.to_timestamp(), 1, w.to_numpy(),
                   where=(w.to_numpy() < 1), alpha=0.15, color='#D62728')
ymid = w.min() + 0.92 * (w.max() - w.min())
ax[0].text(w.index.min().to_timestamp(), ymid, ' discovery', fontsize=9, color='k')
ax[0].text(split.to_timestamp(), ymid, ' validation (held out)', fontsize=9, color='k')
ax[0].set_ylabel('growth of $1')
ax[0].set_title(f'long high {SIG} / short low {SIG},  within lowest {COND}  (h{HZ})')

# drawdown underneath tells you what the Sharpe hides
dd = w / np.maximum.accumulate(w) - 1
axd = ax[0].twinx()
axd.fill_between(dd.index.to_timestamp(), dd.to_numpy(), 0, alpha=0.18,
                 color='#D62728')
axd.set_ylim(dd.min() * 4, 0.001)
axd.set_yticks([])

d = rep[rep['sample'] != 'all']
x = np.arange(len(d))
ax[1].bar(x - 0.2, d.t_nw, 0.4, label='t (Newey-West)', color='#4C78A8')
ax[1].bar(x + 0.2, d.sharpe_ann, 0.4, label='Sharpe (ann.)', color='#F58518')
for y in (-2, 2):
    ax[1].axhline(y, color='r', ls=':', lw=0.9)
ax[1].axhline(0, color='k', lw=0.8)
ax[1].set_xticks(x); ax[1].set_xticklabels(d['sample'])
ax[1].legend(fontsize=8)
ax[1].set_title('discovery vs held-out validation')
plt.tight_layout(); plt.show()

def _line(row, name):
    if not np.isfinite(row.get('t_nw', np.nan)):
        return (f'{name:10s} n/a -- only {row.get("n_quarters", 0)} quarters '
                f'(performance needs >= 8)')
    return (f'{name:10s} t = {row.t_nw:+.2f}   Sharpe {row.sharpe_ann:+.2f}   '
            f'maxDD {row.max_drawdown:+.1%}   n = {int(row.n_quarters)}')

dsc = rep[rep['sample'] == 'discovery'].iloc[0]
val = rep[rep['sample'] == 'validation'].iloc[0]
print(_line(dsc, 'discovery'))
print(_line(val, 'validation'))

if np.isfinite(dsc.t_nw) and np.isfinite(val.t_nw):
    if np.sign(dsc.t_nw) == np.sign(val.t_nw):
        print('\nsign agrees across the split')
    else:
        print('\nSIGN FLIP across the split -- the strategy is dead')
else:
    print(f'\ncannot compare: one side is too short. cfg.split={cfg.split!r}, '
          f'data ends {sp.index.max()} -- move the split earlier to test.')

In [ ]:
# Is bucket 0 special, or would any sum_abs_aw bucket do? And does it hold at h2,
# the tradable horizon under a 45-60 day filing delay?
rows = []
for h in (1, 2, 3):
    for b in range(5):
        s2 = OD.strategy_spread(sq, SIG, COND, bucket=b, n_buckets=5, cfg=cfg,
                                horizon=h)
        if len(s2) < 8:
            continue
        r = OD.report_strategy(s2, cfg, h, f'b{b}')
        for _, row in r.iterrows():
            rows.append(dict(horizon=h, bucket=b, sample=row['sample'],
                             t_nw=row.t_nw, sharpe=row.sharpe_ann,
                             n_q=row.n_quarters))
ctx = pd.DataFrame(rows)
print(f'{SIG} long-short t, by {COND} bucket (0 = lowest) and horizon:\n')
print(ctx[ctx['sample'] == 'all'].pivot_table(index='bucket', columns='horizon',
                                              values='t_nw').round(2).to_string())
print('\ndiscovery / validation at h1:')
print(ctx[(ctx.horizon == 1) & (ctx['sample'] != 'all')]
      .pivot_table(index='bucket', columns='sample', values='t_nw').round(2).to_string())